# Credit Card Default Prediction — Classification (Parts A & B)

**Dataset:** `default_of_credit_card_clients_real.csv`  
**Target:** `default payment next month` (0 = no default, 1 = default)

This notebook implements all ten classification algorithms required by the 23CSE301 Machine Learning Capstone guidelines. Every model uses the same stratified 80:20 split and the same training-only preprocessing pipeline. Random state is set to 42 for reproducibility.

## 1. Imports and reproducibility

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier,
    BaggingClassifier
)
from sklearn.neural_network import MLPClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)
print("Libraries imported. Random state:", RANDOM_STATE)

## 2. Load the specified dataset
The loader checks the notebook's current folder, a local `data/` folder, and the standard uploaded-file location. If none is found, update `DATA_PATH` to the CSV location on your computer.

In [ ]:
TARGET = "default payment next month"
possible_paths = [
    "default_of_credit_card_clients_real.csv",
    os.path.join("data", "default_of_credit_card_clients_real.csv"),
    "/mnt/data/default_of_credit_card_clients_real.csv"
]
DATA_PATH = next((p for p in possible_paths if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find default_of_credit_card_clients_real.csv. "
        "Place it beside this notebook or inside a data/ folder, then update possible_paths."
    )

df = pd.read_csv(DATA_PATH)
# Normalize accidental whitespace in column names without changing their meaning.
df.columns = df.columns.astype(str).str.strip()
if TARGET not in df.columns:
    raise ValueError(f"Expected target column {TARGET!r}. Available columns: {df.columns.tolist()}")
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())


## 3. Dataset audit
Check shape, data types, missing values, duplicates, and target-class distribution before modelling.

In [ ]:
print("Dataset shape:", df.shape)
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
}))
print("Duplicate rows:", int(df.duplicated().sum()))
print("\nTarget distribution:")
display(df[TARGET].value_counts(dropna=False).rename_axis("class").to_frame("count"))
display((df[TARGET].value_counts(normalize=True, dropna=False) * 100).round(2).rename("percentage").to_frame())
if df[TARGET].isna().any():
    raise ValueError("The target contains missing values; inspect and resolve them before training.")


## 4. Exploratory data analysis (EDA)
The following plots cover the target distribution, distributions of all features, a correlation heatmap, and two feature–target relationships. Use the plots to form observations from the actual output rather than assuming a result in advance.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET, color=sns.color_palette("colorblind")[0])
plt.title("Target Distribution: Credit Card Default")
plt.xlabel("Default next month (0 = No, 1 = Yes)")
plt.ylabel("Number of clients")
plt.tight_layout()
plt.show()

feature_cols_for_eda = [c for c in df.columns if c != TARGET and c.upper() != "ID"]
ncols = 4
nrows = int(np.ceil(len(feature_cols_for_eda) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.0))
axes = np.array(axes).reshape(-1)
for ax, col in zip(axes, feature_cols_for_eda):
    sns.histplot(data=df, x=col, bins=30, ax=ax, color=sns.color_palette("colorblind")[0])
    ax.set_title(f"Distribution: {col}")
for ax in axes[len(feature_cols_for_eda):]:
    ax.set_visible(False)
fig.suptitle("Feature Distributions", y=1.01, fontsize=16)
plt.tight_layout()
plt.show()

numeric_for_corr = df.select_dtypes(include=np.number)
plt.figure(figsize=(17, 13))
sns.heatmap(numeric_for_corr.corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Heatmap of Numeric Columns")
plt.tight_layout()
plt.show()

for feature in ["LIMIT_BAL", "AGE"]:
    if feature in df.columns:
        plt.figure(figsize=(7, 4))
        sns.boxplot(data=df, x=TARGET, y=feature, color=sns.color_palette("colorblind")[1])
        plt.title(f"{feature} by Default Class")
        plt.xlabel("Default next month (0 = No, 1 = Yes)")
        plt.ylabel(feature)
        plt.tight_layout()
        plt.show()


### EDA observations — complete after reviewing your plots
- **Target distribution:** use the count plot to describe whether the two classes are balanced or imbalanced.
- **Feature distributions:** note skewed monetary features and unusual values visible in the histograms.
- **Correlation heatmap:** identify strongly correlated variables; correlation does not by itself prove causation.
- **Feature–target plots:** compare the distributions across classes and describe only patterns visible in your plots.

## 5. Cleaning and feature engineering
Remove exact duplicate rows, exclude the ID identifier from predictors, and create aggregate billing/payment features. Outliers are counted with the IQR rule for continuous monetary/age variables; these observations are not automatically deleted because unusually high bills or payments may be legitimate financial behaviour. Imputation and scaling are fitted only on the training data inside pipelines.

In [ ]:
# Remove exact duplicate records, if present.
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Exact duplicate rows removed:", before - len(df))

# Feature engineering: totals and average monthly payment-to-bill ratio.
bill_cols = [c for c in df.columns if c.upper().startswith("BILL_AMT")]
pay_cols = [c for c in df.columns if c.upper().startswith("PAY_AMT")]
if bill_cols:
    df["TOTAL_BILL_AMT"] = df[bill_cols].sum(axis=1)
    df["AVG_BILL_AMT"] = df[bill_cols].mean(axis=1)
if pay_cols:
    df["TOTAL_PAY_AMT"] = df[pay_cols].sum(axis=1)
    df["AVG_PAY_AMT"] = df[pay_cols].mean(axis=1)
if bill_cols and pay_cols:
    df["PAY_TO_BILL_RATIO"] = df["TOTAL_PAY_AMT"] / df["TOTAL_BILL_AMT"].replace(0, np.nan)
    df["PAY_TO_BILL_RATIO"] = df["PAY_TO_BILL_RATIO"].replace([np.inf, -np.inf], np.nan)

print("Engineered features:", [c for c in ["TOTAL_BILL_AMT", "AVG_BILL_AMT", "TOTAL_PAY_AMT", "AVG_PAY_AMT", "PAY_TO_BILL_RATIO"] if c in df.columns])
# IQR outlier audit on selected continuous columns.
continuous_cols = [c for c in ["LIMIT_BAL", "AGE"] + bill_cols + pay_cols if c in df.columns]
outlier_rows = []
for col in continuous_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_rows.append({"Feature": col, "IQR lower bound": low, "IQR upper bound": high,
                         "Outlier count": int(((df[col] < low) | (df[col] > high)).sum())})
display(pd.DataFrame(outlier_rows))
print("Outlier treatment: flagged for audit; not removed automatically because extreme financial values may be valid.")


## 6. Prepare features, target, and one consistent split
`ID` is excluded because it is an identifier. The target is kept separate. The dataset's coded demographic/repayment-status fields are treated as categorical; monetary and age fields are numeric. One-hot encoding, imputation, and scaling are fitted by each pipeline on training data only.

In [ ]:
X = df.drop(columns=[TARGET] + [c for c in df.columns if c.upper() == "ID"], errors="ignore").copy()
y = df[TARGET].astype(int).copy()

# The expected target is binary; stop clearly if the provided file differs.
if set(y.dropna().unique()) != {0, 1}:
    raise ValueError(f"Expected binary target labels 0 and 1; found {sorted(y.dropna().unique().tolist())}")

categorical_candidates = ["SEX", "EDUCATION", "MARRIAGE", "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
categorical_features = [c for c in categorical_candidates if c in X.columns]
numeric_features = [c for c in X.columns if c not in categorical_features]
print("Categorical features:", categorical_features)
print("Numeric features:", len(numeric_features))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train target proportions:\n", y_train.value_counts(normalize=True).sort_index().round(3))
print("Test target proportions:\n", y_test.value_counts(normalize=True).sort_index().round(3))

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


## 7. Classification algorithms — Part A (five models)
Part A includes Logistic Regression, KNN, Gaussian Naive Bayes, Decision Tree, and SVC.

In [ ]:
part_a_models = {
    "Logistic Regression": LogisticRegression(max_iter=1500, class_weight="balanced", random_state=RANDOM_STATE),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
    "Gaussian Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=RANDOM_STATE),
    "Support Vector Machine": SVC(C=1.0, kernel="linear", probability=False, class_weight="balanced", random_state=RANDOM_STATE)
}
print("Part A models:", list(part_a_models))


## 8. Classification algorithms — Part B (five models)
Part B uses the **same dataset, target, train/test split, and preprocessing approach** as Part A: Random Forest, AdaBoost, Gradient Boosting, Bagging with a Decision Tree base estimator, and MLP Neural Network.

In [ ]:
part_b_models = {
    "Random Forest": RandomForestClassifier(n_estimators=160, max_depth=None, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE),
    "Bagging (Decision Tree)": BaggingClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE), n_estimators=60, n_jobs=-1, random_state=RANDOM_STATE),
    "MLP Neural Network": MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", max_iter=250, early_stopping=True, random_state=RANDOM_STATE)
}
print("Part B models:", list(part_b_models))


## 9. Train all ten models and evaluate consistently
Metrics required by the guidelines: accuracy, weighted precision, weighted recall, weighted F1, ROC-AUC, and confusion matrix. ROC-AUC is calculated from positive-class probabilities; all models here support `predict_proba`.

In [ ]:
all_models = {**part_a_models, **part_b_models}
fitted_models = {}
predictions = {}
probabilities = {}
result_rows = []

for name, classifier in all_models.items():
    model = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", classifier)])
    print(f"Training {name} ...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = (model.predict_proba(X_test)[:, 1] if hasattr(model.named_steps["classifier"], "predict_proba") else model.decision_function(X_test))
    fitted_models[name] = model
    predictions[name] = y_pred
    probabilities[name] = y_prob
    result_rows.append({
        "Algorithm": name,
        "Track": "Part A" if name in part_a_models else "Part B",
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(result_rows).sort_values("F1-Score", ascending=False).reset_index(drop=True)
display(results_df.round(4))


## 10. Confusion matrices and classification reports
Confusion matrices show correct and incorrect predictions for each class. Class 1 is the default class, so pay attention to its recall as well as overall accuracy.

In [ ]:
for name in all_models:
    print(f"\n{name} — classification report")
    print(classification_report(y_test, predictions[name], target_names=["No default (0)", "Default (1)"], zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        y_test, predictions[name],
        display_labels=["No default (0)", "Default (1)"],
        cmap="Blues", values_format="d"
    )
    plt.title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()


## 11. Visual comparison of all ten models

In [ ]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
plot_data = results_df.melt(id_vars=["Algorithm", "Track"], value_vars=metrics_to_plot,
                            var_name="Metric", value_name="Score")
plt.figure(figsize=(16, 7))
sns.barplot(data=plot_data, x="Algorithm", y="Score", hue="Metric", palette="colorblind")
plt.title("Comparison of All Ten Classification Algorithms")
plt.xlabel("Algorithm")
plt.ylabel("Score")
plt.xticks(rotation=35, ha="right")
plt.ylim(0, 1)
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 12. Hyperparameter tuning — Random Forest
A small GridSearchCV searches a limited parameter grid using stratified 3-fold cross-validation. The held-out test set is not used to select hyperparameters. Compare the tuned model against the untuned Random Forest on the same test split.

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE))
])
param_grid = {
    "classifier__n_estimators": [100, 160],
    "classifier__max_depth": [None, 12],
    "classifier__min_samples_split": [2]
}
tuning_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
grid_search = GridSearchCV(
    rf_pipeline, param_grid=param_grid, scoring="f1_weighted",
    cv=tuning_cv, n_jobs=-1, verbose=1, refit=True
)
grid_search.fit(X_train, y_train)
tuned_rf = grid_search.best_estimator_
tuned_pred = tuned_rf.predict(X_test)
tuned_prob = tuned_rf.predict_proba(X_test)[:, 1]

def metric_row(name, y_true, y_pred, y_score, track="Tuned"):
    return {
        "Algorithm": name, "Track": track,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_score)
    }

baseline_rf_row = results_df.loc[results_df["Algorithm"] == "Random Forest"].iloc[0]
tuned_rf_row = metric_row("Tuned Random Forest", y_test, tuned_pred, tuned_prob)
tuning_comparison = pd.DataFrame([baseline_rf_row.to_dict(), tuned_rf_row])
print("Best parameters:", grid_search.best_params_)
print("Best CV weighted F1:", round(grid_search.best_score_, 4))
display(tuning_comparison.round(4))
print("Weighted F1 change (tuned - baseline):", round(tuned_rf_row["F1-Score"] - baseline_rf_row["F1-Score"], 4))


## 13. Five-fold cross-validation for the two best baseline models
The two highest weighted-F1 models from the initial ten-model comparison are evaluated with stratified 5-fold cross-validation on the training set. This gives a more stable estimate than relying on one split.

In [ ]:
top_two_names = results_df.head(2)["Algorithm"].tolist()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name in top_two_names:
    cv_scores = cross_val_score(fitted_models[name], X_train, y_train, cv=cv, scoring="f1_weighted", n_jobs=-1)
    cv_rows.append({"Algorithm": name, "CV folds": 5, "Mean weighted F1": cv_scores.mean(), "Std weighted F1": cv_scores.std(),
                    "Fold scores": np.round(cv_scores, 4).tolist()})
display(pd.DataFrame(cv_rows))


## 14. Final consolidated comparison — all ten required algorithms
The required table contains exactly the ten baseline algorithms. The tuned Random Forest is reported separately above so the consolidated ten-model comparison remains easy to interpret.

In [ ]:
final_comparison = results_df[["Algorithm", "Track", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]].copy()
final_comparison = final_comparison.sort_values("F1-Score", ascending=False).reset_index(drop=True)
display(final_comparison.round(4))
best_name = results_df.iloc[0]["Algorithm"]
print("Best baseline model by weighted F1:", best_name)
print("Its test metrics:")
display(results_df.loc[results_df["Algorithm"] == best_name].round(4))
print("\nTuned Random Forest best parameters:", grid_search.best_params_)


## 15. Feature importance (tree-based model)
The Random Forest feature importances are extracted after preprocessing, including one-hot encoded categorical features. These importances indicate how the fitted forest used features; they do not establish causation.

In [ ]:
rf_fitted = fitted_models["Random Forest"]
feature_names = rf_fitted.named_steps["preprocessor"].get_feature_names_out()
importances = rf_fitted.named_steps["classifier"].feature_importances_
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances}).sort_values("Importance", ascending=False).head(20)
plt.figure(figsize=(10, 7))
sns.barplot(data=importance_df, x="Importance", y="Feature", color=sns.color_palette("colorblind")[0])
plt.title("Top 20 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()
display(importance_df.reset_index(drop=True))


## 16. Conclusion and reporting checklist

Complete this section after running the notebook and reviewing the actual metrics and plots:
1. State which baseline model achieved the highest weighted F1 and compare its ROC-AUC.
2. Describe the tuned Random Forest's measured change in weighted F1 and other metrics.
3. Discuss the trade-off between detecting defaults (class-1 recall) and avoiding false alarms (precision).
4. Mention limitations: the results reflect this dataset and split; model scores do not guarantee real-world lending outcomes.

**Guideline coverage:** dataset audit; target and feature distributions; correlation heatmap; two feature–target plots; duplicate/missing/outlier checks; engineered features; stratified 80:20 split; training-only preprocessing; five Part A + five Part B models; accuracy/weighted precision/recall/F1/ROC-AUC; confusion matrices; hyperparameter tuning; five-fold cross-validation for the two best baseline models; consolidated ten-model table; feature importance.